# 18-phase8-growable-foundations

**neuron Phase 8** — structural axis 진입 1차 검증. Phase 4~7 의 결론 (gating function axis 한계)
이후 paradigm 의 다음 axis 로 진입: nn.Parameter 정적 shape 초월.

핵심 가설:
1. **AdamW state 보존 시 학습 continuity** — expansion 직후 loss spike 없이 학습 지속?
2. **function preservation 효과** — zero-init vs random-init 신규 dim 의 loss curve 차이?
3. **반복 expansion 안정성** — 여러 번 expansion 후에도 학습 발산 없이 진행?
4. **AdamW state reset vs 보존** — momentum 보존이 실질적 이익 가져옴?

설계: Growable MLP language model (단순 MLP — Transformer 아님, foundation 검증 우선).
- input: 4 token sliding window → embedding lookup → concat
- hidden: GrowableLinear → GELU → GrowableLinear (hidden width 가 학습 중 확장)
- output: vocab logits
- 매 500 step 마다 hidden_dim 128 → 192 → 256 으로 expansion
- ablation: AdamW state 보존 ON/OFF × init mode {zero, normal}

데이터: TinyShakespeare (char-LM)
시드: [42, 123]
작성일: 2026-05-25
연관: Issue [#55](https://github.com/EinSofINTEREST/GraphLM/issues/55)

## 1. 환경 / device

In [ ]:
import logging
import sys

import torch
import torch.nn.functional as F
from torch import Tensor, nn

import graphlm
from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    iter_random_batches,
    load_tinyshakespeare_text,
)
from graphlm.neuron import GrowableEmbedding, GrowableLayerNorm, GrowableLinear
from graphlm.utils import repo_root, set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

logging.basicConfig(
    level=logging.WARNING, format="%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S"
)
print("python  :", sys.version.split()[0])
print("graphlm :", graphlm.__version__)
print("torch   :", torch.__version__)
print("device  :", DEVICE)

## 2. 실험 설정

In [ ]:
ROOT = repo_root()
DATA_PATH = ROOT / "data" / "tinyshakespeare.txt"
OUT_DIR = ROOT / "runs" / "notebook-neuron-phase8"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123]
CONFIGS = [
    # (state_preserve, init_mode) 4 조합
    (True, "zero"),
    (True, "normal"),
    (False, "zero"),
    (False, "normal"),
]
EMB_DIM = 64  # 입력 embedding dim (고정 — embedding 확장은 별도 검증)
INIT_HIDDEN = 128
EXPAND_DELTA = 64  # 매 expansion 마다 hidden width 증가
EXPAND_STEPS = [500, 1000]  # 이 step 에서 expansion 실행
N_GRAM = 4  # 입력: 4 tokens sliding window

BATCH_SIZE = 32
BLOCK_SIZE = N_GRAM + 1  # input N_GRAM + 1 target
TRAIN_LR = 3e-4
MAX_STEPS = 1500

print(f"SEEDS         = {SEEDS}")
print(f"CONFIGS       = {CONFIGS}")
print(f"INIT_HIDDEN   = {INIT_HIDDEN}, EXPAND_DELTA = {EXPAND_DELTA}, at steps = {EXPAND_STEPS}")
print(f"MAX_STEPS     = {MAX_STEPS}")
print(f"전체 run      = {len(SEEDS) * len(CONFIGS)} = {len(SEEDS) * len(CONFIGS)}")

## 3. 데이터 로드 + n-gram dataset

In [ ]:
text = load_tinyshakespeare_text(DATA_PATH)
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
V = tokenizer.vocab_size
print(f"vocab_size : {V}, n_tokens : {len(dataset):,}")

## 4. Growable MLP 정의

`hidden_dim` 이 학습 중 확장되는 단순 MLP. fc1 / fc2 가 GrowableLinear. ln 도 GrowableLayerNorm.

In [ ]:
# Growable MLP-LM 정의는 src/graphlm/neuron/growable_demo.py 로 이동.
# (.claude/rules/06-code-style.md: 노트북 셀에 함수/클래스 정의 금지)
from graphlm.neuron.growable_demo import GrowableMLPLM, train_growable_mlp

## 5. 학습 루프 + sweep

In [ ]:
runs = {}
for seed in SEEDS:
    for state_preserve, init_mode in CONFIGS:
        key = (seed, state_preserve, init_mode)
        print(f"--- seed={seed}, state_preserve={state_preserve}, init={init_mode} ---")
        runs[key] = train_growable_mlp(
            dataset=dataset,
            vocab_size=V,
            seed=seed,
            state_preserve=state_preserve,
            init_mode=init_mode,
            emb_dim=EMB_DIM,
            init_hidden=INIT_HIDDEN,
            expand_delta=EXPAND_DELTA,
            expand_steps=EXPAND_STEPS,
            n_gram=N_GRAM,
            batch_size=BATCH_SIZE,
            lr=TRAIN_LR,
            max_steps=MAX_STEPS,
            device=DEVICE,
        )
        print(f"  done: final_loss={runs[key]['final_loss']:.4f}")
        if str(DEVICE).startswith("cuda"):
            torch.cuda.empty_cache()

## 6. 결과 표 — config x seed

In [ ]:
import statistics

print(
    f"{'state_preserve':>15}  {'init':>7}  {'seed':>5}  "
    f"{'expand@500 loss spike':>22}  {'expand@1000 loss spike':>22}  {'final_loss':>11}"
)
print("-" * 100)
for state_preserve, init_mode in CONFIGS:
    for seed in SEEDS:
        r = runs[(seed, state_preserve, init_mode)]
        # spike = loss[step+1] - loss[step-1] (즉시 변화)
        losses = r["losses"]
        s500 = losses[500] - losses[499] if len(losses) > 500 else 0.0
        s1000 = losses[1000] - losses[999] if len(losses) > 1000 else 0.0
        print(
            f"{str(state_preserve):>15}  {init_mode:>7}  {seed:>5}  "
            f"{s500:>+22.4f}  {s1000:>+22.4f}  {r['final_loss']:>11.4f}"
        )

print()
print("=== 집계 (config 별 mean final_loss) ===")
for state_preserve, init_mode in CONFIGS:
    fls = [runs[(s, state_preserve, init_mode)]["final_loss"] for s in SEEDS]
    print(
        f"  state_preserve={state_preserve}, init={init_mode}: "
        f"mean={statistics.mean(fls):.4f}, range={max(fls) - min(fls):.4f}"
    )

## 7. function preservation 직접 검증

zero-init expansion 직후 forward 가 *수치적으로* 불변인지 sanity check.
별도 mini-test: 학습 안 한 model 로 expansion 전후 비교.

In [ ]:
set_seed(0)
sanity_model = GrowableMLPLM(V, EMB_DIM, INIT_HIDDEN, N_GRAM).to(DEVICE)
sanity_model.eval()
x = torch.randint(0, V, (4, N_GRAM)).to(DEVICE)
with torch.no_grad():
    out_before = sanity_model(x).clone()
    sanity_model.expand_hidden(EXPAND_DELTA, optimizer=None, init="zero")
    out_after_zero = sanity_model(x)

# zero-init 의 경우 LN 의 새 dim 이 0 입력을 정규화하면서 약간 영향을 줄 수 있음 —
# 단, fc2.expand_in 의 새 column 이 0 이라 logits 에 기여 = 0 → output 변화 미미해야
max_diff_zero = (out_before - out_after_zero).abs().max().item()
print(f"zero-init: max |diff| = {max_diff_zero:.6f}")

set_seed(0)
sanity2 = GrowableMLPLM(V, EMB_DIM, INIT_HIDDEN, N_GRAM).to(DEVICE)
sanity2.eval()
with torch.no_grad():
    out_before2 = sanity2(x).clone()
    sanity2.expand_hidden(EXPAND_DELTA, optimizer=None, init="normal")
    out_after_normal = sanity2(x)
max_diff_normal = (out_before2 - out_after_normal).abs().max().item()
print(f"normal-init: max |diff| = {max_diff_normal:.6f}")
print()
print("→ zero-init 의 max |diff| 가 normal-init 보다 훨씬 작아야 function preservation 입증.")

## 8. 시각화 — loss curve + width 진화

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# (a) loss curves — config 별 mean over seeds, smoothed
window = 30
colors = {
    (True, "zero"): "#2ca02c",
    (True, "normal"): "#1f77b4",
    (False, "zero"): "#ff7f0e",
    (False, "normal"): "#d62728",
}
for state_preserve, init_mode in CONFIGS:
    seed_curves = []
    for seed in SEEDS:
        losses = runs[(seed, state_preserve, init_mode)]["losses"]
        smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
        seed_curves.append(smoothed)
    arr = np.array(seed_curves)
    steps = np.arange(window - 1, window - 1 + arr.shape[1])
    mean = arr.mean(axis=0)
    std = arr.std(axis=0, ddof=1)
    label = f"preserve={state_preserve}, init={init_mode}"
    ax1.plot(steps, mean, color=colors[(state_preserve, init_mode)], lw=1.5, label=label)
    ax1.fill_between(
        steps, mean - std, mean + std, color=colors[(state_preserve, init_mode)], alpha=0.15
    )

for s in EXPAND_STEPS:
    ax1.axvline(s, color="gray", linestyle="--", lw=0.8, alpha=0.5)
ax1.set_ylabel(f"loss (smoothed window={window})")
ax1.set_title("Phase 8 — Growable MLP-LM: loss curve × config (mean ± σ over 2 seeds)")
ax1.legend(loc="upper right", fontsize=8)
ax1.grid(alpha=0.3)

# (b) hidden width 진화 (모든 run 동일하므로 첫 run 만)
key0 = (SEEDS[0], CONFIGS[0][0], CONFIGS[0][1])
widths = runs[key0]["widths"]
ax2.plot(range(len(widths)), widths, color="black", lw=1.5, label="hidden_dim")
for s in EXPAND_STEPS:
    ax2.axvline(s, color="gray", linestyle="--", lw=0.8, alpha=0.5)
ax2.set_xlabel("step")
ax2.set_ylabel("hidden_dim")
ax2.set_title("hidden width 진화")
ax2.legend(loc="lower right", fontsize=8)
ax2.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(OUT_DIR / "growable_mlp.png", dpi=120)
plt.show()

## 결과 요약 / Phase 9 권장 방향

확인 포인트:
- §6 의 expansion spike — state_preserve=True + zero-init 조합에서 spike 최소?
- §6 의 final_loss 비교 — AdamW state 보존이 실제 이익?
- §7 의 zero-init function preservation — max |diff| 가 거의 0?
- §8 의 loss curve — expansion 지점에서 다른 config 가 발산?

**판정 시나리오**:
- **A. state_preserve + zero-init 명확 우위** ⭐
  - foundation 검증 완료. Phase 9: 본 모듈들을 활용한 GrowableTransformer 구현
- **B. config 간 큰 차이 없음**
  - small task / small model 에서는 expansion 효과 미미. Phase 9: 더 큰 task 에서 재검증 또는 GrowableTransformer 로 직진
- **C. state reset 도 충분히 회복 가능**
  - AdamW state 보존의 이익 제한적. Phase 9: 그래도 GrowableTransformer 진행 (구조 자체가 paradigm 의 핵심)

**참고 자료**:
- Phase 7 결과: https://www.notion.so/36be8b70b7aa817c9c26feb21c302a25
- Phase 8+ 후보 (foundations 다음 phase 들): https://www.notion.so/36be8b70b7aa8142bebdc9706c64ed52